In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Как выглядит кросс-валидация на k-фолдов:

Фолд 1: [ОБУЧЕНИЕ] [ОБУЧЕНИЕ] [ОБУЧЕНИЕ] [ОБУЧЕНИЕ] [ВАЛ]  → метрика₁

Фолд 2: [ОБУЧЕНИЕ] [ОБУЧЕНИЕ] [ОБУЧЕНИЕ] [ВАЛ] [ОБУЧЕНИЕ] → метрика₂

Фолд 3: [ОБУЧЕНИЕ] [ОБУЧЕНИЕ] [ВАЛ] [ОБУЧЕНИЕ] [ОБУЧЕНИЕ] → метрика₃

Фолд 4: [ОБУЧЕНИЕ] [ВАЛ] [ОБУЧЕНИЕ] [ОБУЧЕНИЕ] [ОБУЧЕНИЕ] → метрика₄

Фолд 5: [ВАЛ] [ОБУЧЕНИЕ] [ОБУЧЕНИЕ] [ОБУЧЕНИЕ] [ОБУЧЕНИЕ] → метрика₅

Итоговая метрика = среднее (метрика₁ … метрика₅)

In [2]:
df = pd.read_csv(r"Advertising.csv")

df.head()

X = df.drop('sales', axis=1)
y = df['sales']

from sklearn.model_selection import train_test_split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = .3, random_state=101) # можно указать поменьше, обучение все равно на всех данных, которые в трейн

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
from sklearn.linear_model import Ridge

In [9]:
model = Ridge(alpha=100)

In [10]:
from sklearn.model_selection import cross_val_score # тоже moel_select, потому что отвечает за разбиения

In [11]:
scores = cross_val_score(model, X_train, y_train, scoring='neg_mean_squared_error', cv=5) 
'''
estimator = модель машинного обучения estimate = оценивать
X = X_train
y = y_train
cv = разбиения
scoring = метрика
https://scikit-learn.org/stable/modules/model_evaluation.html
чем значение больше = тем лучше, поэтому метрика mse отрицательна будет
'''


'\nestimator = модель машинного обучения estimate = оценивать\nX = X_train\ny = y_train\ncv = разбиения\nscoring = метрика\nhttps://scikit-learn.org/stable/modules/model_evaluation.html\nчем значение больше = тем лучше, поэтому метрика mse отрицательна будет\n'

In [12]:
'''
значение метрики для каждой из пяти итераций cv
'''
scores

array([ -9.32552967,  -4.9449624 , -11.39665242,  -7.0242106 ,
        -8.38562723])

In [13]:
abs(scores.mean()) # среднее

np.float64(8.215396464543609)

In [14]:
model = Ridge(alpha=4)


In [15]:
scores = cross_val_score(model, X_train, y_train, scoring='neg_mean_squared_error', cv=5) 

In [16]:
scores

array([-3.22317553, -1.48538914, -5.51133561, -2.16850422, -4.45168262])

In [17]:
abs(scores.mean()) # среднее

np.float64(3.3680174242211693)

## Ответ от ии, зачем нужна кросс-валидация

Пример из жизни
Представьте, что вы тестируете ученика по 5 билетам (кросс-валидация) — средняя оценка 4.0. А если случайно вытянуть «лёгкий» билет (hold-out), он может получить 5.0. Но настоящий уровень ученика — 4.0. Точно так же hold-out может дать слишком оптимистичную оценку качества модели.

Когда это критично?
Малый датасет (как ваш Advertising, ~200 строк) — случайность разбиения сильно влияет. Кросс-валидация надёжнее.

Сильный разброс данных — например, если в одном из фолдов окажутся все выбросы.

Что делать?
Используйте cross_val_score для оценки стабильности (посмотрите на разброс метрик по фолдам — может быть, один фолд сильно выбивается).

Для финального сравнения двух моделей лучше полагаться на среднюю по кросс-валидации, а не на одно hold-out разбиение.

Если hold-out показывает лучший результат, чем средний по кросс-валидации — скорее всего, вам просто повезло с разбиением. Не доверяйте этому «лучшему» результату.

----
Почему hold-out (70/30) может показывать лучший результат, чем средняя по кросс-валидации?
Размер тестовой выборки больше (30% vs 14%)
Оценка на 30% данных обычно имеет меньшую дисперсию, чем оценка на 14% (хотя она всё равно зависит от одного разбиения). Но «лучший» результат может быть просто из-за того, что конкретные 30% тестовых данных оказались «лёгкими» для модели.

Кросс-валидация усредняет «хорошие» и «плохие» фолды
Даже если на 4 фолдах ошибка низкая, на 5-м фолде может быть выброс — среднее станет хуже, чем удачное hold-out разбиение. Hold-out может случайно попасть на «лёгкий» кусок (который соответствует одному из хороших фолдов) и показать завышенное качество.

Hold-out не штрафует за переобучение под конкретный состав данных
Вы обучаете модель один раз на 70% данных. Модель может подстроиться под особенности именно этих 70%. При кросс-валидации модель обучается 5 раз на разных 56% данных — это более строгий тест на обобщающую способность.

метод крос вал скор не выполнил обучение моодели

посмотретьь как ведет себя на разных разбиениях данных

In [18]:
from sklearn.metrics import mean_squared_error

In [19]:
model.fit(X_train, y_train)

mse = mean_squared_error(y_test, model.predict(X_test))
mse

2.391740156472669